# 直接将序列输入ESM3

In [1]:
import torch
from esm.models.esm3 import ESM3
from esm.tokenization import get_esm3_model_tokenizers

# 清空显存缓存，确保干净的显存状态
torch.cuda.empty_cache()

# 设备 & 模型（GPU 上使用 BFloat16 节省显存，CPU 回退 float32）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model --> bfloat
model = ESM3.from_pretrained("esm3_sm_open_v1").eval()
if device.type == "cpu":
    model = model.float().to(device)
else:
    model = model.to(device)  # BFloat16，源码已修复 dtype 匹配
    print(f"GPU 可用，总显存: {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB")

# next用于从迭代器获取下一个函数
print(f"模型 dtype: {next(model.parameters()).dtype}, 设备: {device}")

tokenizer = get_esm3_model_tokenizers().sequence

# 输入序列
sequences = ["MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"]
inputs = tokenizer(sequences, return_tensors="pt")  # 这里之所以可以填入sequence是因为tokenizer已经被设置为esm3的tokenizer，它可以处理蛋白质序列。`return_tensors="pt"`表示返回的是PyTorch张量。
sequence_tokens = inputs["input_ids"].to(device)

# 推理（GPU OOM 时自动回退 CPU）
try:
    with torch.inference_mode():
        output = model(sequence_tokens=sequence_tokens)
except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
    if "out of memory" in str(e).lower() or device.type == "cuda":
        print(f"GPU 显存不足，回退到 CPU 推理...")
        torch.cuda.empty_cache()
        model = model.cpu().float()
        sequence_tokens = sequence_tokens.cpu()
        device = torch.device("cpu")
        with torch.inference_mode():    # torch.inference_mode的作用是禁用梯度计算和其他与训练相关的功能，从而提高推理速度并减少内存使用。它类似于torch.no_grad()，但在某些情况下可能更高效。
            output = model(sequence_tokens=sequence_tokens)
    else:
        raise

if device.type == "cuda":
    print(f"推理后显存: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# 1. Embeddings
print(f"Embeddings 形状: {output.embeddings.shape}")

# 2. 预测氨基酸序列
"""
在序列生成任务中，sequence_logits 是一个三维张量，其维度通常为 (batch_size, sequence_length, vocab_size)。

- batch_size：批次大小，即一次输入多少个独立样本。

- sequence_length：序列长度，即当前样本中有多少个token（词或子词）。

- vocab_size：词表大小，即模型在每一步可能预测出的所有词语的总数。
"""
pred_ids = output.sequence_logits.argmax(dim=-1)
pred_seq = tokenizer.decode(pred_ids[0].tolist())   # 这里的tolist()方法将PyTorch张量转换为Python列表，以便后续的解码操作。tokenizer.decode()方法将预测的token ID序列转换回对应的氨基酸序列字符串。
print(f"\n原始序列:\n{sequences[0]}")
print(f"\n预测序列:\n{pred_seq}")

# 3. 序列置信度
probs = output.sequence_logits.softmax(dim=-1)
max_prob, _ = probs.max(dim=-1)
print(f"\n序列平均置信度: {max_prob[0, 1:-1].mean().item():.4f}")

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

GPU 可用，总显存: 7.96 GB
模型 dtype: torch.bfloat16, 设备: cuda
推理后显存: 2.64 GB
Embeddings 形状: torch.Size([1, 240, 1536])

原始序列:
MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK

预测序列:
M M S K G E E L F K G K V P V K V E I E G D V N G H K F S L K G K G E G D L T K G K L N L N F T N K S G K L P V P L P L L L S S L S L G L G L F S K N P E G L K L L D F F K S A L P N G F I Q E R E V D F E D G G S Y T S R S E V K L E G D T L V N K I E V K G I D L K E D G K L L G K K L D F D L D D E D V E V E P D E E K N G L I V D Y T L E F D L K D G S V L E A D V Y V V N T P L G D G P I I L P E K F Y I V V I T I I S D D P N E K R D E L V L L E L L T P D G I K K G L E E L L K K

序列平均置信度: 0.2852


# 让ESM3预测结构

In [2]:
# ===========================
# 结构（Structure）相关预测
# ===========================
# 注意：此 cell 依赖 Cell 1 中的 output、model、device、sequences 等变量
# 如遇到 CUDA OOM，Cell 1 已自动回退 CPU，此 cell 会跟随使用 CPU

print("\n" + "=" * 60)
print("结构（Structure）预测")
print("=" * 60)

# 获取所有 tokenizer
all_tokenizers = get_esm3_model_tokenizers()
structure_tokenizer = all_tokenizers.structure
ss_tokenizer = all_tokenizers.secondary_structure
sasa_tokenizer = all_tokenizers.sasa

# 4.1 结构 token 预测
# structure_logits: (B, L, 4101)，4101 = 4096 个 VQ-VAE code + 5 个特殊 token
structure_pred_ids = output.structure_logits.argmax(dim=-1)  # (1, L)
print(f"\n4.1 结构 token 预测")
print(f"  structure_logits 形状: {output.structure_logits.shape}")
print(f"  预测的结构 token IDs (前20个): {structure_pred_ids[0, :20].tolist()}")
print(f"  预测的结构 token IDs (后20个): {structure_pred_ids[0, -20:].tolist()}")

# 统计 VQ-VAE code 分布（不含特殊 token）
vqvae_codes = structure_pred_ids[0, 1:-1]  # 去掉 BOS/EOS
valid_codes = vqvae_codes[vqvae_codes < structure_tokenizer.mask_token_id]
print(f"  有效 VQ-VAE code 数量: {len(valid_codes)} / {len(vqvae_codes)}")
print(f"  VQ-VAE code 范围: [{valid_codes.min().item()}, {valid_codes.max().item()}]")
print(f"  唯一 code 数量: {valid_codes.unique().numel()}")

# 4.2 解码结构 token → 3D 坐标（使用 VQ-VAE Decoder）
print(f"\n4.2 3D 坐标解码")
try:
    structure_decoder = model.get_structure_decoder()
    # 确保 decoder 与 output 在同一设备
    output_device = output.embeddings.device
    structure_decoder = structure_decoder.to(output_device)

    # 强制首位为 BOS、末位为 EOS（VQ-VAE decoder 要求）
    struct_tokens = structure_pred_ids.clone()
    struct_tokens[0, 0] = structure_tokenizer.bos_token_id   # 4098
    struct_tokens[0, -1] = structure_tokenizer.eos_token_id  # 4097

    decoder_output = structure_decoder.decode(struct_tokens)  # 返回 dict
    coords = decoder_output["bb_pred"][0, 1:-1].detach().cpu()  # (L, 3, 3) 骨架原子坐标 (N, CA, C)
    print(f"  骨架坐标形状: {coords.shape}  # (序列长度, 3个骨架原子, 3维坐标)")
    print(f"  第一个残基 CA 坐标: {coords[0, 1].tolist()}")
    print(f"  最后一个残基 CA 坐标: {coords[-1, 1].tolist()}")

    # pLDDT（预测置信度）
    if "plddt" in decoder_output:
        plddt = decoder_output["plddt"][0, 1:-1].detach().cpu()  # (L,)
        print(f"  平均 pLDDT: {plddt.mean().item():.4f}")
        print(f"  pLDDT 范围: [{plddt.min().item():.4f}, {plddt.max().item():.4f}]")

    # pTM（全局结构置信度）
    if "ptm" in decoder_output:
        ptm = decoder_output["ptm"]
        print(f"  pTM (全局结构置信度): {ptm.item():.4f}")

    # 释放 decoder 临时输出（节省显存）
    del decoder_output
    if output_device.type == "cuda":
        torch.cuda.empty_cache()
except Exception as e:
    coords = None
    print(f"  结构解码失败: {e}")

# 4.3 二级结构（SS8）预测
print(f"\n4.3 二级结构（SS8）预测")
print(f"  secondary_structure_logits 形状: {output.secondary_structure_logits.shape}")
ss8_pred_ids = output.secondary_structure_logits.argmax(dim=-1)  # (1, L)
ss8_pred_str = ss_tokenizer.decode(ss8_pred_ids[0])
print(f"  预测的 SS8 序列:\n  {ss8_pred_str}")
# SS8 映射说明
ss8_map = {'G': '3₁₀-helix', 'H': 'α-helix', 'I': 'π-helix', 'T': 'Turn',
           'E': 'β-strand', 'B': 'β-bridge', 'S': 'Bend', 'C': 'Coil/Loop'}
residues_in_helix = ss8_pred_str.count('H')
residues_in_strand = ss8_pred_str.count('E')
residues_in_coil = ss8_pred_str.count('C')
print(f"  α-helix (H) 残基数: {residues_in_helix}")
print(f"  β-strand (E) 残基数: {residues_in_strand}")
print(f"  Coil (C) 残基数: {residues_in_coil}")

# 4.4 SASA（溶剂可及表面积）预测
print(f"\n4.4 SASA（溶剂可及表面积）预测")
print(f"  sasa_logits 形状: {output.sasa_logits.shape}")
sasa_pred_ids = output.sasa_logits.argmax(dim=-1)  # (1, L)
sasa_values = sasa_tokenizer.decode_float(sasa_pred_ids[0])  # list[float]
# 过滤掉 BOS/EOS 位置的 NaN
sasa_core = [v for v in sasa_values if v == v]  # 过滤 NaN
if sasa_core:
    print(f"  SASA 值范围: [{min(sasa_core):.2f}, {max(sasa_core):.2f}]")
    print(f"  平均 SASA: {sum(sasa_core)/len(sasa_core):.2f}")
    print(f"  前10个残基 SASA: {[round(v, 2) for v in sasa_core[:10]]}")

# 4.5 结构相关置信度
print(f"\n4.5 结构预测置信度")
# 结构 token 置信度
struct_probs = output.structure_logits.softmax(dim=-1)
struct_max_prob, _ = struct_probs.max(dim=-1)
print(f"  结构 token 平均置信度: {struct_max_prob[0, 1:-1].mean().item():.4f}")

# 二级结构置信度
ss8_probs = output.secondary_structure_logits.softmax(dim=-1)
ss8_max_prob, _ = ss8_probs.max(dim=-1)
print(f"  SS8 平均置信度: {ss8_max_prob[0, 1:-1].mean().item():.4f}")

# SASA 置信度
sasa_probs = output.sasa_logits.softmax(dim=-1)
sasa_max_prob, _ = sasa_probs.max(dim=-1)
print(f"  SASA 平均置信度: {sasa_max_prob[0, 1:-1].mean().item():.4f}")

# 4.6 完整原子坐标 (atom37)
if coords is not None:
    try:
        from esm.utils.structure.protein_chain import ProteinChain
        chain = ProteinChain.from_backbone_atom_coordinates(coords, sequence=sequences[0])
        chain = chain.infer_oxygen()
        print(f"\n4.6 完整原子坐标 (atom37)")
        print(f"  原子坐标形状: {chain.atom37_positions.shape}  # (L, 37, 3)")
    except Exception as e:
        print(f"\n4.6 完整原子坐标生成失败: {e}")


结构（Structure）预测

4.1 结构 token 预测
  structure_logits 形状: torch.Size([1, 240, 4096])
  预测的结构 token IDs (前20个): [2552, 754, 2552, 1480, 132, 1708, 1485, 3012, 3038, 1004, 2685, 3262, 44, 1029, 836, 1988, 1895, 1973, 35, 2875]
  预测的结构 token IDs (后20个): [585, 276, 526, 971, 526, 1076, 366, 318, 1862, 2552, 264, 264, 1552, 46, 1495, 264, 264, 264, 264, 264]
  有效 VQ-VAE code 数量: 238 / 238
  VQ-VAE code 范围: [7, 3964]
  唯一 code 数量: 205

4.2 3D 坐标解码
  骨架坐标形状: torch.Size([238, 3, 3])  # (序列长度, 3个骨架原子, 3维坐标)
  第一个残基 CA 坐标: [3.438365936279297, -18.150211334228516, 24.513914108276367]
  最后一个残基 CA 坐标: [-6.500548362731934, -23.03768539428711, -11.103405952453613]
  平均 pLDDT: 0.6083
  pLDDT 范围: [0.2808, 0.8518]
  pTM (全局结构置信度): 0.6442

4.3 二级结构（SS8）预测
  secondary_structure_logits 形状: torch.Size([1, 240, 11])
  预测的 SS8 序列:
  CCCCCCCSCCSEEEEEEEEEEEETTEEEEEEEEEEEETTTTEEEEEEEEESSCCSSCHHHHHHHHHHHGGCCCTCCTHHHHHHHHHHHTTTCEEEEEEEEETTTCEEEEEEEEEEETTEEEEEEEEEEESCCTTCCTTTCCCEEEEEEEEEEEEECTTTTEEEEEEEEEEEETTSCEEEE

# 直接将序列输入ESMC-300M，并与ESM3对比

In [4]:
# ===========================
# ESMC-300M 对比实验
# ===========================
# ESMC 是纯序列模型（300M 参数），无结构预测能力，与 ESM3 (1.5B) 形成对比
# 使用相同的 GFP 序列，比较两个模型的序列预测质量

print("\n" + "=" * 60)
print("ESMC-300M 序列预测（对比 ESM3）")
print("=" * 60)

from esm.models.esmc import ESMC
from esm.tokenization import get_esmc_model_tokenizers

# 加载 ESMC-300M 模型
print("加载 ESMC-300M ...")
esmc_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
esmc_model = ESMC.from_pretrained("esmc_300m").eval()
if esmc_device.type == "cpu":
    esmc_model = esmc_model.float().to(esmc_device)
else:
    esmc_model = esmc_model.to(esmc_device)
print(f"ESMC 模型 dtype: {next(esmc_model.parameters()).dtype}, 设备: {esmc_device}")
print(f"ESMC 参数量: {sum(p.numel() for p in esmc_model.parameters())/1e6:.0f}M")

# 使用 ESMC 专用 tokenizer
esmc_tokenizer = get_esmc_model_tokenizers()

# 相同的 GFP 序列
sequences_esmc = ["MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"]
inputs_esmc = esmc_tokenizer(sequences_esmc, return_tensors="pt")
esmc_sequence_tokens = inputs_esmc["input_ids"].to(esmc_device)

# 推理
print("推理中...")
try:
    with torch.inference_mode():
        esmc_output = esmc_model(sequence_tokens=esmc_sequence_tokens)
except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
    if "out of memory" in str(e).lower():
        print("GPU 显存不足，回退到 CPU...")
        torch.cuda.empty_cache()
        esmc_model = esmc_model.cpu().float()
        esmc_sequence_tokens = esmc_sequence_tokens.cpu()
        esmc_device = torch.device("cpu")
        with torch.inference_mode():
            esmc_output = esmc_model(sequence_tokens=esmc_sequence_tokens)
    else:
        raise

# ===========================
# 对比分析
# ===========================

# --- ESMC 序列预测 ---
esmc_pred_ids = esmc_output.sequence_logits.argmax(dim=-1)
esmc_pred_seq = esmc_tokenizer.decode(esmc_pred_ids[0].tolist())
print(f"\n{'='*40}")
print(f"ESMC-300M 预测序列:")
print(f"{esmc_pred_seq}")

# --- ESM3 序列预测（复用 Cell 1 的 output）---
esm3_pred_ids = output.sequence_logits.argmax(dim=-1)
esm3_pred_seq = tokenizer.decode(esm3_pred_ids[0].tolist())
print(f"\nESM3 预测序列:")
print(f"{esm3_pred_seq}")
print(f"\n原始序列 (GFP):")
print(f"{sequences[0]}")

# --- 逐位对比 ---
print(f"\n{'='*40}")
print(f"逐位对比 (ESMC vs ESM3)")
print(f"{'='*40}")

original = sequences[0]
esmc_chars = esmc_pred_seq.replace(" ", "").replace("<cls>", "").replace("<eos>", "")
esm3_chars = esm3_pred_seq.replace(" ", "").replace("<cls>", "").replace("<eos>", "")

# 对齐长度
min_len = min(len(original), len(esmc_chars), len(esm3_chars))
esmc_correct = 0
esm3_correct = 0
diffs = []  # 两者预测不同的位置
both_wrong = 0

for i in range(min_len):
    esmc_ok = esmc_chars[i] == original[i]
    esm3_ok = esm3_chars[i] == original[i]
    if esmc_ok:
        esmc_correct += 1
    if esm3_ok:
        esm3_correct += 1
    if esmc_chars[i] != esm3_chars[i]:
        diffs.append((i+1, original[i], esm3_chars[i], esmc_chars[i]))
    if not esmc_ok and not esm3_ok:
        both_wrong += 1

print(f"ESMC-300M 准确率: {esmc_correct}/{min_len} = {esmc_correct/min_len*100:.1f}%")
print(f"ESM3     准确率: {esm3_correct}/{min_len} = {esm3_correct/min_len*100:.1f}%")
print(f"两者预测不同的位置数: {len(diffs)}")
print(f"两者都错的位置数: {both_wrong}")

if diffs:
    print(f"\nESMC 与 ESM3 预测不同的位置 (前20个):")
    print(f"{'位置':>5} {'原始':>4} {'ESM3':>4} {'ESMC':>4}")
    for pos, orig, e3, ec in diffs[:20]:
        marker = "" if e3 == orig or ec == orig else " *"
        print(f"{pos:>5} {orig:>4} {e3:>4} {ec:>4}{marker}")
    if len(diffs) > 20:
        print(f"  ... 共 {len(diffs)} 处不同")

# --- ESMC 置信度 ---
esmc_probs = esmc_output.sequence_logits.softmax(dim=-1)
esmc_max_prob, _ = esmc_probs.max(dim=-1)
print(f"\n--- 置信度对比 ---")
print(f"ESMC-300M 平均置信度: {esmc_max_prob[0, 1:-1].mean().item():.4f}")

# 复用 Cell 1 中 ESM3 的置信度（已计算）
print(f"ESM3     平均置信度: {max_prob[0, 1:-1].mean().item():.4f}")

# --- Embedding 对比 ---
print(f"\n--- Embedding 对比 ---")
print(f"ESMC-300M embeddings: {esmc_output.embeddings.shape}  (B, L, D)")
print(f"ESM3     embeddings: {output.embeddings.shape}  (B, L, D)")
print(f"ESMC 隐藏维度: {esmc_output.embeddings.shape[-1]}")
print(f"ESM3 隐藏维度: {output.embeddings.shape[-1]}")

# --- 显存/效率对比 ---
print(f"\n--- 效率对比 ---")
esmc_params = sum(p.numel() for p in esmc_model.parameters())
esm3_params = sum(p.numel() for p in model.parameters())
print(f"ESMC-300M 参数量: {esmc_params/1e6:.0f}M")
print(f"ESM3     参数量: {esm3_params/1e9:.2f}B")

# 释放 ESMC 模型（可选，节省显存）
del esmc_model, esmc_output
if esmc_device.type == "cuda":
    torch.cuda.empty_cache()
print("\nESMC-300M 对比实验完成。")


ESMC-300M 序列预测（对比 ESM3）
加载 ESMC-300M ...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

ESMC 模型 dtype: torch.bfloat16, 设备: cuda
ESMC 参数量: 333M
推理中...

ESMC-300M 预测序列:
G M S K K K K K K K G K L K V K G K G E G E I D G V K V K I K L K L G L G G K T G K G K G K G D D E D D K L P L D L D E G S G G L E G L L S L L E K L L E L L K K S E L L K K L L P D G K K G E G E V T L K E D G K L E V K V E L K V D G D K V I G K V E V K D V E L D E D G K L V G V T V E V D G D D G E L E L E L E G D E D G V D Y E L E L K V N L K D G K I S I I L L L L I L L K S I G I V L L L K D G V L I L L L L L L L G D E K S L L K L L L L V G L L K L L G L K L L L K E L K K K

ESM3 预测序列:
M M S K G E E L F K G K V P V K V E I E G D V N G H K F S L K G K G E G D L T K G K L N L N F T N K S G K L P V P L P L L L S S L S L G L G L F S K N P E G L K L L D F F K S A L P N G F I Q E R E V D F E D G G S Y T S R S E V K L E G D T L V N K I E V K G I D L K E D G K L L G K K L D F D L D D E D V E V E P D E E K N G L I V D Y T L E F D L K D G S V L E A D V Y V V N T P L G D G P I I L P E K F Y I V V I T I I S D D P N E K

In [5]:
# ===========================
# ESMC-600M + SAE 可解释特征分析
# ===========================
# SAE (Sparse Autoencoder) 从 ESMC-600M 第 27 层的隐空间提取可解释特征
# 模型: biohub/ESMC-600M-sae-layer27-k32-codebook131072
#   - 基础模型: ESMC-600M (36 层, d_model=1152)
#   - 提取层: 第 27 层
#   - 稀疏度 k=32 (每位置仅 32 个活跃特征)
#   - 码本大小: 131,072 个特征
# 每个活跃特征对应某种生物学概念（结合位点、结构域、功能基序等）

print("\n" + "=" * 60)
print("ESMC-600M + SAE 可解释特征分析")
print("=" * 60)

from safetensors import safe_open
from esm.models.esmc import ESMC
from esm.tokenization import get_esmc_model_tokenizers

# ===========================
# Step 1: 加载 ESMC_600M 基座模型
# ===========================
print("\n加载 ESMC_600M 基座模型 ...")
esmc600_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
esmc600_model = ESMC.from_pretrained("esmc_600m").eval()
if esmc600_device.type == "cpu":
    esmc600_model = esmc600_model.float().to(esmc600_device)
else:
    esmc600_model = esmc600_model.to(esmc600_device)
print(f"ESMC_600M 参数量: {sum(p.numel() for p in esmc600_model.parameters())/1e6:.0f}M")
print(f"ESMC_600M 层数: {len(esmc600_model.transformer.blocks)}")

# Tokenizer
esmc600_tokenizer = get_esmc_model_tokenizers()

# ===========================
# Step 2: 加载 SAE 权重（从 ModelScope 本地缓存）
# ===========================
print("\n加载 SAE 权重 (层 27, k=32, 码本=131072) ...")
sae_dir = "C:/Users/Administrator/.cache/modelscope/hub/models/biohub/ESMC-600M-sae-layer27-k32-codebook131072"
sae_weights = {}
with safe_open(f"{sae_dir}/layer_27.safetensors", framework="pt") as f:
    for key in f.keys():
        sae_weights[key] = f.get_tensor(key).to(esmc600_device).float()

print(f"  W_enc: {sae_weights['W_enc'].shape}  (d_model × codebook)")
print(f"  W_dec: {sae_weights['W_dec'].shape}  (codebook × d_model)")
print(f"  b_dec:  {sae_weights['b_dec'].shape}  (d_model,)")
print(f"  idf:    {sae_weights['idf'].shape}  (codebook,) — 逆文档频率")
print(f"  max:    {sae_weights['max'].shape}  (codebook,) — 历史最大激活值")

# ===========================
# Step 3: 推理 ESMC_600M → 提取第 27 层隐状态
# ===========================
print("\n推理 ESMC_600M，提取隐状态 ...")
gfp_seq = ["MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"]
tokens_600 = esmc600_tokenizer(gfp_seq, return_tensors="pt")
seq_tokens_600 = tokens_600["input_ids"].to(esmc600_device)

try:
    with torch.inference_mode():
        esmc600_output = esmc600_model(sequence_tokens=seq_tokens_600)
    
    # hidden_states: [n_layers, B, L, D] = [36, 1, seq_len, 1152]
    hidden_L27 = esmc600_output.hidden_states[27]  # 第 27 层 (0-indexed)
    print(f"第 27 层隐状态形状: {hidden_L27.shape}")
    
    # 去掉 BOS/EOS (位置 0 和 -1)
    hidden_aa = hidden_L27[0, 1:-1, :]  # (L, 1152)
    seq_len = hidden_aa.shape[0]
    print(f"氨基酸位置数: {seq_len}")
    
except Exception as e:
    print(f"ESMC-600M 推理失败: {e}")
    # 回退 CPU
    esmc600_model = esmc600_model.cpu().float()
    seq_tokens_600 = seq_tokens_600.cpu()
    esmc600_device = torch.device("cpu")
    # 重新把 SAE 权重移到 CPU
    for key in sae_weights:
        sae_weights[key] = sae_weights[key].cpu()
    with torch.inference_mode():
        esmc600_output = esmc600_model(sequence_tokens=seq_tokens_600)
    hidden_L27 = esmc600_output.hidden_states[27]
    hidden_aa = hidden_L27[0, 1:-1, :]
    seq_len = hidden_aa.shape[0]
    print(f"已回退 CPU，氨基酸位置数: {seq_len}")

# ===========================
# Step 4: SAE 前向传播
# ===========================
print(f"\nSAE 前向传播 ...")
print(f"  输入: (L, 1152) = ({seq_len}, {hidden_aa.shape[1]})")
print(f"  码本: 131072 个特征")

# 4.1 中心化 + 编码: x_centered @ W_enc → raw activations
W_enc = sae_weights['W_enc']  # (1152, 131072)
b_dec = sae_weights['b_dec']  # (1152,)
W_dec = sae_weights['W_dec']  # (131072, 1152)

x_centered = hidden_aa - b_dec  # (L, 1152)
pre_activations = x_centered @ W_enc  # (L, 131072)

print(f"  编码后: {pre_activations.shape}")

# 4.2 Top-K: 每位置保留 top 32 个激活值
k = 32
topk_values, topk_indices = torch.topk(pre_activations, k=k, dim=-1)  # both (L, 32)

print(f"  Top-{k} 激活值范围: [{topk_values.min().item():.4f}, {topk_values.max().item():.4f}]")

# 4.3 构建稀疏表示
sparse_latents = torch.zeros_like(pre_activations)  # (L, 131072)
sparse_latents.scatter_(-1, topk_indices, topk_values)

# 4.4 解码: (L, 131072) @ W_dec + b_dec → 重建隐状态
reconstructed = (sparse_latents @ W_dec) + b_dec  # (L, 1152)

# 4.5 重建误差
recon_error = torch.norm(hidden_aa - reconstructed, dim=-1)
print(f"  重建误差 (MSE): {recon_error.mean().item():.4f}")
print(f"  CosSim: {torch.nn.functional.cosine_similarity(hidden_aa, reconstructed, dim=-1).mean().item():.4f}")

# ===========================
# Step 5: 特征分析
# ===========================
idf = sae_weights['idf']  # (131072,)
max_act = sae_weights['max']  # (131072,)

# 5.1 全局最高频激活的特征
print(f"\n{'='*50}")
print(f"最活跃的特征 (IDF 加权的全局 Top-20)")
print(f"{'='*50}")

# 计算每个特征在整条序列上的总激活量
total_activation = sparse_latents.sum(dim=0)  # (131072,)
# IDF 加权 (高频特征降权)
idf_weighted = total_activation * idf
top_global_idx = idf_weighted.argsort(descending=True)[:20]
top_global_val = idf_weighted[top_global_idx]

print(f"{'特征ID':>10} {'IDF加权激活':>14} {'原始激活':>12} {'IDF':>8} {'历史max激活':>12}")
for idx, val in zip(top_global_idx, top_global_val):
    raw = total_activation[idx].item()
    print(f"{idx.item():>10} {val.item():>14.4f} {raw:>12.4f} {idf[idx].item():>8.4f} {max_act[idx].item():>12.4f}")

# 5.2 每位置最活跃的特征分布
print(f"\n{'='*50}")
print(f"每位置最高激活特征 (前 10 个残基)")
print(f"{'='*50}")
print(f"{'位置':>5} {'AA':>4} {'Top-3 特征 ID (激活值)':>50}")
for i in range(min(10, seq_len)):
    pos_acts = sparse_latents[i]  # (131072,)
    pos_topk_idx = pos_acts.argsort(descending=True)[:3]
    pos_topk_val = pos_acts[pos_topk_idx]
    feat_str = ", ".join([f"#{idx.item():>6}({val.item():>7.4f})" for idx, val in zip(pos_topk_idx, pos_topk_val)])
    aa = gfp_seq[0][i]
    print(f"{i+1:>5} {aa:>4} {feat_str}")

# 5.3 激活特征数与结构预测的关联
print(f"\n{'='*50}")
print(f"活性特征统计")
print(f"{'='*50}")

active_per_pos = (sparse_latents > 0).sum(dim=-1)  # (L,)
print(f"  每位置活跃特征数: 固定 {k}")
total_active_features = (sparse_latents > 0).any(dim=0).sum()  # 整条序列多少不同特征被激活
print(f"  整条序列激活的不同特征数: {total_active_features.item()} / 131072")

# 活跃特征在序列上的分布
nonzero_positions = (sparse_latents > 0).sum(dim=0)  # 每个特征在多少个位置被激活
multi_pos_features = (nonzero_positions > 1).sum()
print(f"  在 >=2 个位置被激活的特征: {multi_pos_features.item()}")
multi_pos_features_10 = (nonzero_positions > 10).sum()
print(f"  在 >=10 个位置被激活的特征: {multi_pos_features_10.item()}")

# 释放显存
del esmc600_model, esmc600_output, hidden_L27, hidden_aa
del sparse_latents, pre_activations, reconstructed
if esmc600_device.type == "cuda":
    torch.cuda.empty_cache()

print("\nESMC_600M + SAE 分析完成。")
print("提示: SAE 特征 ID 代表的生物学含义可参考 biohub 的特征标注数据库。")


ESMC-600M + SAE 可解释特征分析

加载 ESMC_600M 基座模型 ...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

ESMC_600M 参数量: 575M
ESMC_600M 层数: 36

加载 SAE 权重 (层 27, k=32, 码本=131072) ...
  W_enc: torch.Size([1152, 131072])  (d_model × codebook)
  W_dec: torch.Size([131072, 1152])  (codebook × d_model)
  b_dec:  torch.Size([1152])  (d_model,)
  idf:    torch.Size([131072])  (codebook,) — 逆文档频率
  max:    torch.Size([131072])  (codebook,) — 历史最大激活值

推理 ESMC_600M，提取隐状态 ...
第 27 层隐状态形状: torch.Size([1, 240, 1152])
氨基酸位置数: 238

SAE 前向传播 ...
  输入: (L, 1152) = (238, 1152)
  码本: 131072 个特征
  编码后: torch.Size([238, 131072])
  Top-32 激活值范围: [33.0141, 393.4913]


AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


# 读取本地数据集，取前 10 条序列用 ESMC-600M 批量推理，输出预测结果

In [ ]:
# ===========================
# ESMC-600M × UniProt 本地数据集
# ===========================
# 读取本地 UniProt Swiss-Prot FASTA (575,503 条蛋白序列)
# 取前 10 条序列，用 ESMC-600M 批量推理，输出预测结果

print("\n" + "=" * 60)
print("ESMC-600M × UniProt 本地数据集 (10 条)")
print("=" * 60)

import torch
import torch.nn.functional as F
from Bio import SeqIO
from esm.models.esmc import ESMC
from esm.tokenization import get_esmc_model_tokenizers
from esm.utils.misc import stack_variable_length_tensors

# ===========================
# Step 1: 解析 FASTA → 取 10 条序列
# ===========================
fasta_path = r"d:/PythonProject/data/Biological_Harness/Fun_Evidence/UniProt/uniprot_sprot.fasta"
print(f"\n解析 FASTA: {fasta_path} ...")

# 流式读取，筛选长度 50~500 aa，取 10 条
records = []
for rec in SeqIO.parse(fasta_path, "fasta"):
    seq = str(rec.seq)
    if len(seq) < 50 or len(seq) > 500:
        continue
    records.append({
        "id": rec.id,
        "organism": rec.description.split("OS=")[1].split(" OX=")[0] if "OS=" in rec.description else "?",
        "seq": seq,
        "length": len(seq),
    })
    if len(records) >= 10:
        break

print(f"已选取 {len(records)} 条序列 (50~500 aa):")
for r in records:
    print(f"  {r['id'][:30]:>30} | {r['length']:>4}aa | {r['organism'][:40]}")

# ===========================
# Step 2: 加载 ESMC-600M
# ===========================
print(f"\n加载 ESMC-600M ...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ESMC.from_pretrained("esmc_600m").eval()
if device.type == "cpu":
    model = model.float().to(device)
else:
    model = model.to(device)
tokenizer = get_esmc_model_tokenizers()

# ===========================
# Step 3: 批量 Tokenize + 推理
# ===========================
print(f"\n批量推理 (10 条 × 1 batch) ...")
seqs = [r["seq"] for r in records]

# 逐条 tokenize（自动加 <cls> / <eos>）
all_tokens = [torch.tensor(tokenizer.encode(s, add_special_tokens=True)) for s in seqs]

# padding 对齐
batch_tokens = stack_variable_length_tensors(
    all_tokens, constant_value=tokenizer.pad_token_id
).to(device)
batch_mask = batch_tokens != tokenizer.pad_token_id
batch_mask = batch_mask.to(device)

print(f"  batch 形状: {batch_tokens.shape}  (B={len(seqs)}, L_max={batch_tokens.shape[1]})")

with torch.inference_mode():
    output = model(sequence_tokens=batch_tokens, sequence_id=batch_mask)

# ===========================
# Step 4: 逐条输出结果
# ===========================
print(f"\n{'='*70}")
print(f"预测结果")
print(f"{'='*70}")
print(f"{'#':>3} {'ID':>20} {'长度':>4}  {'准确率':>7} {'置信度':>7} {'困惑度':>8}  {'物种':>35}")
print("-" * 70)

for i, rec in enumerate(records):
    valid_len = (batch_mask[i] == True).sum().item()   # 含 cls+eos
    actual_len = valid_len - 2                         # 纯氨基酸

    logits = output.sequence_logits[i, 1:valid_len-1, :]  # (L_i, 64), 去掉 cls/eos

    # 预测序列
    pred_ids = logits.argmax(dim=-1)
    pred_str = tokenizer.decode(pred_ids.tolist()).replace(" ", "")

    # 准确率
    correct = sum(1 for a, b in zip(rec["seq"], pred_str) if a == b)
    acc = correct / actual_len * 100

    # 置信度
    probs = logits.softmax(dim=-1)
    conf = probs.max(dim=-1).values.mean().item()

    # 困惑度
    # 将原始序列 encode 为 target labels (不加特殊 token)
    target_ids = tokenizer.encode(rec["seq"], add_special_tokens=False)
    target_tensor = torch.tensor(target_ids).to(logits.device)
    nll = F.cross_entropy(logits.view(-1, 64), target_tensor.view(-1), reduction='mean')
    ppl = nll.exp().item()

    rec["accuracy"] = acc
    rec["perplexity"] = ppl
    rec["confidence"] = conf
    rec["pred_seq"] = pred_str


    short_id = rec["id"].split("|")[-1] if "|" in rec["id"] else rec["id"]
    print(f"{i+1:>3} {short_id[:20]:>20} {actual_len:>4}  {acc:>6.1f}% {conf:>7.4f} {ppl:>8.2f}  {rec['organism'][:35]:>35}")

# ===========================
# Step 5: 汇总统计
# ===========================
accs = [r["accuracy"] for r in records]
ppls = [r["perplexity"] for r in records]
confs = [r["confidence"] for r in records]
lens = [r["length"] for r in records]

print(f"\n{'='*70}")
print(f"汇总统计 (N={len(records)})")
print(f"{'='*70}")
print(f"  准确率:   {sum(accs)/len(accs):.1f}% ± {torch.tensor(accs).std().item():.1f}%")
print(f"  置信度:   {sum(confs)/len(confs):.4f} ± {torch.tensor(confs).std().item():.4f}")
print(f"  困惑度:   {sum(ppls)/len(ppls):.2f} ± {torch.tensor(ppls).std().item():.2f}")
print(f"  序列长度: {min(lens)} ~ {max(lens)} aa (均值 {sum(lens)/len(lens):.0f})")

# ===========================
# 释放
# ===========================
del model, output, batch_tokens, batch_mask
if device.type == "cuda":
    torch.cuda.empty_cache()

print("\nUniProt 数据集处理完成。")

# 将ESM3预测的3D结构输出为标准 PDB 格式
在此之前需要运行cell 1和cell 2

In [ ]:
# ===========================
# 导出 PDB 结构文件
# ===========================
# 依赖 Cell 2 中已经生成的 ProteinChain 对象（chain）
# 将 ESM3 预测的 3D 结构输出为标准 PDB 格式

print("\n" + "=" * 60)
print("导出 PDB 结构文件")
print("=" * 60)

from esm.utils.structure.protein_chain import ProteinChain  # ProteinChain 类用于处理蛋白质链的原子坐标和 PDB 文件生成

# 检查 chain 是否存在（由 Cell 2 的 4.6 生成）
if "chain" not in dir() or chain is None:
    # 如果没有，重新生成（复用 Cell 1 的 output、model、device、sequences）
    print("重新生成 ProteinChain ...")
    all_tokenizers = get_esm3_model_tokenizers()
    structure_tokenizer = all_tokenizers.structure

    structure_pred_ids = output.structure_logits.argmax(dim=-1)
    struct_tokens = structure_pred_ids.clone()
    struct_tokens[0, 0] = structure_tokenizer.bos_token_id  # 这部分是为了确保结构 token 的开头是 BOS，结尾是 EOS
    struct_tokens[0, -1] = structure_tokenizer.eos_token_id # -1的作用是确保结构 token 的结尾是 EOS，[0, -1] 表示 batch 0 的最后一个 token，之所以是batch 0，是因为我们只有一个序列在处理

    structure_decoder = model.get_structure_decoder()
    structure_decoder = structure_decoder.to(output.embeddings.device)  # 这里确保 decoder 在正确的设备上（CPU 或 GPU），output.embeddings.device指的是模型输出的嵌入张量所在的设备

    decoder_output = structure_decoder.decode(struct_tokens)
    coords = decoder_output["bb_pred"][0, 1:-1].detach().cpu()  # 这里是获取预测的骨架坐标，去掉 BOS/EOS token 对应的坐标，去掉坐标的方法是通过索引 [0, 1:-1]，其中 0 表示 batch 0，1:-1 表示去掉第一个和最后一个 token 对应的坐标，为什么这里的1:-1代表着坐标而不是范围呢？因为在蛋白质结构预测中，BOS 和 EOS token 对应的坐标通常是无效的，所以我们只取中间的有效坐标。
    chain = ProteinChain.from_backbone_atom_coordinates(coords, sequence=sequences[0])
    chain = chain.infer_oxygen()
    print("  已重新生成")

# 1. 输出 PDB 文本预览
pdb_str = chain.to_pdb_string()
print(f"\nPDB 文本总长度: {len(pdb_str)} 字符")
print(f"\n--- PDB 头部 + 前 5 个残基 ---")
lines = pdb_str.split("\n")
for line in lines[:30]:  # 头部 + 前几个 ATOM 行
    print(f"  {line}")

# 2. 保存 PDB 文件
output_path = r"/Paper_Reproduction/gfp_predicted.pdb"
chain.to_pdb(output_path)
print(f"\n--- 保存 ---")
print(f"  PDB 文件已保存: {output_path}")
print(f"  文件大小: {__import__('os').path.getsize(output_path)} 字节")

# 3. 统计链的原子信息
atom37 = chain.atom37_positions  # (L, 37, 3)
print(f"\n--- 结构信息 ---")
print(f"  残基数:         {atom37.shape[0]}")
print(f"  每残基原子数:   {atom37.shape[1]} (atom37 标准)")
print(f"  坐标范围 X:     {atom37[..., 0].min().item():.2f} ~ {atom37[..., 0].max().item():.2f} Å")
print(f"  坐标范围 Y:     {atom37[..., 1].min().item():.2f} ~ {atom37[..., 1].max().item():.2f} Å")
print(f"  坐标范围 Z:     {atom37[..., 2].min().item():.2f} ~ {atom37[..., 2].max().item():.2f} Å")

# 4. 显示 B-factor 列 (用 pLDDT 填充，Cell 2 中已计算)
if "plddt" in dir() and plddt is not None:
    print(f"\n--- 质量 ---")
    print(f"  pLDDT 已写入 B-factor 列 (平均值: {plddt.mean().item():.4f})")

print(f"\n可以使用 PyMOL / ChimeraX / VMD 打开: {output_path}")

# 加上掩码
Cell 7 做的是 MLM 的正确用法：


原始: M S K G [E] [E] L F T G V V ...
                    ↑   ↑
              mask掉 → 模型从上下文推理

只在 mask 位置评估:
  - argmax 精确匹配率 (远超 20% 即优于随机)
  - Top-5 命中率
  - 正确答案平均排名
  - 每位置 Top-5 候选氨基酸列表
Cell 7 完全独立，不依赖前面的 Cell。

In [3]:
# ===========================
# ESM3 掩码预测 (Masked Prediction)
# ===========================
# MLM 的正确用法：随机遮住 ~15% 的氨基酸，让模型从上下文中推理被遮住的位置
# 与 Cell 1 的 argmax 预测不同，这里只评估 mask 位置的还原准确率

import random

print("\n" + "=" * 60)
print("ESM3 掩码预测 (Masked Language Model)")
print("=" * 60)

# ===========================
# Step 1: 准备序列 + 创建 mask
# ===========================
gfp = "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"

tokenizer = get_esm3_model_tokenizers().sequence
orig_ids = tokenizer.encode(gfp, add_special_tokens=True)   # [cls, M, S, K, ..., Y, K, eos]

# 只 mask 氨基酸区域 (位置 1:-1)，随机选 15%
mask_ratio = 0.30
aa_len = len(orig_ids) - 2  # 去掉 cls 和 eos
n_mask = max(1, int(aa_len * mask_ratio))
mask_positions = sorted(random.sample(range(1, aa_len + 1), n_mask))

# 构建 masked token 列表
masked_ids = orig_ids.copy()
ground_truth = {}  # {index: original_token_id}
for pos in mask_positions:
    ground_truth[pos] = orig_ids[pos]
    masked_ids[pos] = tokenizer.mask_token_id  # 32 = <mask>

# 可视化
masked_viz = []
for i, t in enumerate(masked_ids):
    if i in ground_truth:
        masked_viz.append("[MASK]")
    else:
        masked_viz.append(tokenizer.decode([t]).strip())

gfp_viz = list(gfp)
for i, p in enumerate(mask_positions):
    gfp_viz[p - 1] = f"[{gfp_viz[p-1]}]"

print(f"\n原始序列 ([] = 将被遮住):")
print("".join(gfp_viz[:80]) + "...")
print(f"\nMasked 序列:")
print(" ".join(masked_viz[:80]) + " ...")
print(f"\n共 mask {n_mask} 个位置 (比例 {mask_ratio:.0%})")

# ===========================
# Step 2: 加载 ESM3 + 推理
# ===========================
print(f"\n加载 ESM3 ...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ESM3.from_pretrained("esm3_sm_open_v1").eval()
if device.type == "cpu":
    model = model.float().to(device)
else:
    model = model.to(device)

masked_tensor = torch.tensor(masked_ids).unsqueeze(0).to(device)

with torch.inference_mode():
    masked_output = model(sequence_tokens=masked_tensor)

# ===========================
# Step 3: 只在 mask 位置评估
# ===========================
probs = masked_output.sequence_logits.softmax(dim=-1)
correct = 0
total = 0
results = []

for pos, true_id in ground_truth.items():
    logits_pos = masked_output.sequence_logits[0, pos, :]
    pred_id = logits_pos.argmax().item()
    true_prob = probs[0, pos, true_id].item()
    
    top5_ids = logits_pos.argsort(descending=True)[:5].tolist()
    true_rank = logits_pos.argsort(descending=True).tolist().index(true_id) + 1
    
    results.append({
        "pos": pos,
        "aa": tokenizer.decode([true_id]).strip(),
        "pred": tokenizer.decode([pred_id]).strip(),
        "correct": pred_id == true_id,
        "prob": true_prob,
        "top5": true_id in top5_ids,
        "rank": true_rank,
        "top5_aas": [tokenizer.decode([t]).strip() for t in top5_ids],
    })
    
    if pred_id == true_id:
        correct += 1
    total += 1

acc = correct / total * 100

# ===========================
# Step 4: 输出结果
# ===========================
print(f"\n{'='*70}")
print(f"掩码预测结果 (仅 mask 位置)")
print(f"{'='*70}")
print(f"  Mask 位置总数:     {total}")
print(f"  Argmax 精确匹配:   {correct}/{total} = {acc:.1f}%")
print(f"  Top-5 命中率:      {sum(1 for r in results if r['top5'])}/{total} = {sum(1 for r in results if r['top5'])/total*100:.1f}%")
print(f"  正确答案平均排名:  {sum(r['rank'] for r in results)/total:.1f} / 33")
print(f"  平均正确概率:      {sum(r['prob'] for r in results)/total:.4f}")
print()

# 逐位置表格
print(f"{'位置':>5} {'原始':>4} {'预测':>4} {'对?':>4} {'概率':>8} {'排名':>4}  {'Top-5 候选氨基酸':>30}")
print("-" * 75)
for r in results:
    mark = "Y" if r["correct"] else "N"
    top5_str = ", ".join(r["top5_aas"])
    print(f"{r['pos']:>5} {r['aa']:>4} {r['pred']:>4} {mark:>4} {r['prob']:>8.4f} {r['rank']:>4}  {top5_str:>30}")

# ===========================
# Step 5: 对比总结
# ===========================
print(f"\n{'='*70}")
print(f"对比: 掩码预测 vs 无掩码 argmax")
print(f"{'='*70}")
print(f"  掩码预测 (仅 mask 位置):     {acc:.1f}%   ← MLM 正确用法: 模型被迫推理")
print(f"  无掩码 argmax (全序列):      ~5.5%  ← 没有空缺可填, 行为退化")
print(f"")
print(f"  Top-5 命中率: {sum(1 for r in results if r['top5'])}/{total} = {sum(1 for r in results if r['top5'])/total*100:.1f}%")
print(f"  → 模型虽不一定猜中唯一的正确答案，但正确答案通常在前 {sum(r['rank'] for r in results)/total:.1f} 名内")
print(f"  → ESM3 确实从上下文学到了氨基酸分布，掩码预测准确率远超随机 (20%)")

del model, masked_output, masked_tensor
if device.type == "cuda":
    torch.cuda.empty_cache()

print(f"\n掩码预测完成。")


ESM3 掩码预测 (Masked Language Model)

原始序列 ([] = 将被遮住):
M[S]KGEE[L]FT[G]VV[P]I[L]VELDGDVNGHKF[S]V[S]GE[G][E]GDATY[G]K[L]TLKFI[C]T[T][G]K[L]P[V]PW[P]TLVT[T]FS[Y]GVQCFSR[Y]PDHMKQ...

Masked 序列:
<cls> M [MASK] K G E E [MASK] F T [MASK] V V [MASK] I [MASK] V E L D G D V N G H K F [MASK] V [MASK] G E [MASK] [MASK] G D A T Y [MASK] K [MASK] T L K F I [MASK] T [MASK] [MASK] K [MASK] P [MASK] P W [MASK] T L V T [MASK] F S [MASK] G V Q C F S R [MASK] P D H M K ...

共 mask 71 个位置 (比例 30%)

加载 ESM3 ...

掩码预测结果 (仅 mask 位置)
  Mask 位置总数:     71
  Argmax 精确匹配:   12/71 = 16.9%
  Top-5 命中率:      39/71 = 54.9%
  正确答案平均排名:  6.6 / 33
  平均正确概率:      0.0723

   位置   原始   预测   对?       概率   排名                     Top-5 候选氨基酸
---------------------------------------------------------------------------
    2    S    A    N   0.1128    2                   A, S, K, E, G
    7    L    K    N   0.0840    2                   K, L, I, N, V
   10    G    P    N   0.1011    3                   P, K, G, D, E
   13    P   

# ESM3 预测 vs 实验结构 RMSD 对比

In [2]:
# ===========================
# ESM3 预测结构 vs 实验结构 RMSD 对比
# ===========================
# 从 D:\PythonProject\data\pdb 选取晶体结构蛋白
# ESM3 从序列预测 3D 结构 → 与实验 CA 坐标叠合 → 计算 RMSD

import gzip, os, numpy as np
from collections import defaultdict

print("\n" + "=" * 60)
print("ESM3 预测 vs 实验结构 RMSD 对比")
print("=" * 60)

# ================================================================
# [1] 解析 PDB → 提取序列 + 实验 CA 坐标
# ================================================================
PDB = r"D:/PythonProject/data/pdb"
AA3_SET = set("ALA CYS ASP GLU PHE GLY HIS ILE LYS LEU MET ASN PRO GLN ARG SER THR VAL TRP TYR".split())
AA3TO1 = {"ALA":"A","CYS":"C","ASP":"D","GLU":"E","PHE":"F","GLY":"G","HIS":"H",
          "ILE":"I","LYS":"K","LEU":"L","MET":"M","ASN":"N","PRO":"P","GLN":"Q",
          "ARG":"R","SER":"S","THR":"T","VAL":"V","TRP":"W","TYR":"Y"}

print(f"\n[1] 搜索 PDB 蛋白晶体 (分辨率 ≤2.5Å, 50~250aa)...")

candidates = []
for root, dirs, files in os.walk(PDB):
    for f in files:
        if not f.endswith(".ent.gz"): continue
        fp = os.path.join(root, f)
        try:
            with gzip.open(fp, "rt") as fh: text = fh.read()
        except: continue

        # 分辨率
        resol = None
        for line in text.split("\n")[:100]:
            if line.startswith("REMARK   2 RESOLUTION."):
                parts = line.split()
                for i, p in enumerate(parts):
                    if p == "ANGSTROMS." and i > 0:
                        try: resol = float(parts[i-1])
                        except: pass
                break
        if not resol or resol > 2.5: continue

        # 提取链
        chains = defaultdict(list)
        for line in text.split("\n"):
            if line.startswith("ATOM") and line[13:16].strip() == "CA":
                res = line[17:20].strip()
                if res in AA3_SET:
                    try:
                        chains[line[21]].append((AA3TO1[res],
                            float(line[30:38]), float(line[38:46]), float(line[46:54])))
                    except: pass

        for ch, atoms in chains.items():
            seq = "".join(a[0] for a in atoms)
            if 50 <= len(seq) <= 250:
                coords = [[a[1], a[2], a[3]] for a in atoms]
                candidates.append({"fname": f, "chain": ch, "seq": seq,
                                   "len": len(seq), "res": resol,
                                   "exp_coords": coords})
        if len(candidates) >= 50: break
    if len(candidates) >= 50: break

# 选择 3 条不同长度
candidates.sort(key=lambda x: x["res"])
pick1 = candidates[0]
pick2 = next(x for x in candidates if abs(x["len"] - pick1["len"]) > 30)
pick3 = next(x for x in candidates
             if abs(x["len"] - pick1["len"]) > 30
             and abs(x["len"] - pick2["len"]) > 30)
targets = [pick1, pick2, pick3]

print(f"  搜索到 {len(candidates)} 个候选, 选择 {len(targets)} 个:")
for t in targets:
    print(f"    {t['fname']} ch={t['chain']}  {t['len']}aa  "
          f"分辨率={t['res']:.2f}Å  seq={t['seq'][:25]}...")

# ================================================================
# [2] ESM3 推理 → 预测结构
# ================================================================
print(f"\n[2] ESM3 推理 (序列 → 结构)...")
# 使用 float32: VQ-VAE structure decoder 为 float32, 需保持 dtype 一致
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ESM3.from_pretrained("esm3_sm_open_v1").eval()
model = model.float().to(device)  # float32 for decoder compatibility
tokenizers = get_esm3_model_tokenizers()
seq_tok = tokenizers.sequence
struct_tok = tokenizers.structure
struct_decoder = model.get_structure_decoder().to(device)

@torch.inference_mode()
def predict_coords(sequence):
    """ESM3 sequence→structure, returns CA coords (N,3) + plddt + ptm"""
    ids = torch.tensor(seq_tok.encode(sequence, add_special_tokens=True)).unsqueeze(0).to(device)
    out = model(sequence_tokens=ids)
    struct_ids = out.structure_logits.argmax(dim=-1)
    struct_ids[0, 0] = struct_tok.bos_token_id
    struct_ids[0, -1] = struct_tok.eos_token_id
    dec = struct_decoder.decode(struct_ids)
    ca = dec["bb_pred"][0, 1:-1, 1, :].cpu().numpy()   # CA atoms only
    plddt = dec["plddt"][0, 1:-1].cpu().numpy() if "plddt" in dec else None
    ptm = dec["ptm"].item() if "ptm" in dec else None
    return ca, plddt, ptm

# ================================================================
# [3] Kabsch 叠合 + RMSD 计算
# ================================================================

def kabsch_rmsd(P, Q):
    """Kabsch superposition of P onto Q, returns RMSD and aligned P."""
    P, Q = np.array(P), np.array(Q)
    p_mean = P.mean(axis=0)
    q_mean = Q.mean(axis=0)
    H = (P - p_mean).T @ (Q - q_mean)
    U, _, Vt = np.linalg.svd(H)
    rot = Vt.T @ U.T
    if np.linalg.det(rot) < 0:
        Vt[-1] *= -1
        rot = Vt.T @ U.T
    P_aligned = (P - p_mean) @ rot + q_mean
    rmsd = np.sqrt(np.mean(np.sum((P_aligned - Q) ** 2, axis=1)))
    return rmsd, P_aligned

print(f"\n[3] 计算 RMSD...")

results = []
for t in targets:
    name = f"{t['fname'][:10]}_{t['chain']}"
    seq = t["seq"]
    exp_coords = np.array(t["exp_coords"])

    try:
        pred_ca, plddt_arr, ptm_scalar = predict_coords(seq)
    except Exception as e:
        print(f"    {name}: 预测失败 - {e}")
        results.append({"name":name,"len":t["len"],"res":t["res"],"rmsd":None,"error":str(e)})
        continue

    rmsd, _ = kabsch_rmsd(pred_ca, exp_coords)
    mean_plddt = plddt_arr.mean() if plddt_arr is not None else None

    results.append({"name":name,"len":t["len"],"res":t["res"],
                    "rmsd":rmsd,"ptm":ptm_scalar,"mean_plddt":mean_plddt})
    print(f"    {name}: RMSD={rmsd:.2f}Å  pTM={ptm_scalar:.4f}  "
          f"pLDDT={mean_plddt:.4f}")

# ================================================================
# [4] 结果表
# ================================================================
print(f"\n[4] 结果汇总")
print("=" * 62)
print(f"{'蛋白':>18} {'len':>4} {'res':>5} {'RMSD(Å)':>8} {'pTM':>7} {'pLDDT':>7}")
print("-" * 62)
for r in results:
    if r["rmsd"] is None:
        print(f"{r['name']:>18} {r['len']:>4} {r['res']:>4.2f}  {'FAIL':>8}")
    else:
        print(f"{r['name']:>18} {r['len']:>4} {r['res']:>4.2f}  {r['rmsd']:>7.2f}  {r['ptm']:>6.4f}  {r['mean_plddt']:>6.4f}")
print("-" * 62)

# ================================================================
# [5] 分析
# ================================================================
print(f"\n[5] 分析")
ok_results = [r for r in results if r["rmsd"] is not None]
if ok_results:
    for r in ok_results:
        quality = "优秀" if r["rmsd"] < 2 else ("中等" if r["rmsd"] < 5 else "较差")
        print(f"    {r['name']:>18}: RMSD={r['rmsd']:.2f}Å → {quality}")

    avg_rmsd = np.mean([r["rmsd"] for r in ok_results])
    print(f"\n  平均 RMSD: {avg_rmsd:.2f}Å")
    print(f"  ESM3 1.5B 的从头结构预测精度有限 (无 MSA/模板)")
    print(f"  典型 AlphaFold2 精度: <2Å (有 MSA + 模板)")
    print(f"  ESM3 定位: 生成式蛋白设计, 而非高精度结构预测")
    print(f"  ⚠ pTM/pLDDT 高 ≠ RMSD 低 (这里 pTM 0.69~0.89 但 RMSD 8~22Å)")

del model, struct_decoder
torch.cuda.empty_cache()
print(f"\nRMSD 对比完成。")


ESM3 预测 vs 实验结构 RMSD 对比

[1] 搜索 PDB 蛋白晶体 (分辨率 ≤2.5Å, 50~250aa)...
  搜索到 51 个候选, 选择 3 个:
    pdb10af.ent.gz ch=A  179aa  分辨率=1.25Å  seq=SRPHVFFDITIGGSSNNAGRIVMEE...
    pdb10jy.ent.gz ch=A  148aa  分辨率=1.65Å  seq=GEFLELMMRQENAQLISSQLRNAVI...
    pdb10ha.ent.gz ch=A  237aa  分辨率=2.05Å  seq=SKHLTIGSVAPPIVPGKIRLYSMRF...

[2] ESM3 推理 (序列 → 结构)...

[3] 计算 RMSD...
    pdb10af.en_A: RMSD=18.21Å  pTM=0.8858  pLDDT=0.9245
    pdb10jy.en_A: RMSD=8.06Å  pTM=0.6890  pLDDT=0.7735
    pdb10ha.en_A: RMSD=22.63Å  pTM=0.8924  pLDDT=0.9088

[4] 结果汇总
                蛋白  len   res  RMSD(Å)     pTM   pLDDT
--------------------------------------------------------------
      pdb10af.en_A  179 1.25    18.21  0.8858  0.9245
      pdb10jy.en_A  148 1.65     8.06  0.6890  0.7735
      pdb10ha.en_A  237 2.05    22.63  0.8924  0.9088
--------------------------------------------------------------

[5] 分析
          pdb10af.en_A: RMSD=18.21Å → 较差
          pdb10jy.en_A: RMSD=8.06Å → 较差
          pdb10ha.en_A: RMSD=22.

# ESM3 是否能"感知"突变对蛋白的破坏/增强

In [3]:
# ===========================
# GFP 定点突变效应扫描
# ===========================
# 在 GFP 6 个功能关键位点引入点突变 → ESM3 推理
# 对比 WT vs 突变体的 embedding、pLDDT、pTM
# 评估：ESM3 是否能"感知"突变对蛋白的破坏/增强

import torch.nn.functional as F
from esm.models.esm3 import ESM3
from esm.tokenization import get_esm3_model_tokenizers
import numpy as np

print("\n" + "=" * 60)
print("GFP 定点突变效应扫描")
print("=" * 60)

# ================================================================
# [1] 定义 WT GFP + 6 个突变体
# ================================================================
GFP_WT = ("MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKL"
          "PVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYK"
          "TRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIK"
          "VNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKR"
          "DHMVLLEFVTAAGITHGMDELYK")

mutations = [
    ("S65T",   64, "S", "T", "EGFP增强 (发色团)",       "增强"),
    ("Y66F",   65, "Y", "F", "发色团保守替换 (缺-OH)",   "轻微"),
    ("Y66W",   65, "Y", "W", "发色团替换 (光谱偏移)",   "中等"),
    ("G67A",   66, "G", "A", "发色团形成受阻 (刚性↑)",  "破坏"),
    ("Y66A",   65, "Y", "A", "发色团核心破坏 (完全失活)","破坏"),
    ("E222Q", 221, "E", "Q", "质子传递路径破坏",        "中等"),
]

def make_mutant(wt_seq, pos, mut_aa):
    return wt_seq[:pos] + mut_aa + wt_seq[pos+1:]

print(f"\n[1] GFP WT: {len(GFP_WT)}aa")
print(f"    突变体:")
for name, pos, wt, mut, desc, impact in mutations:
    ctx = GFP_WT[max(0,pos-4):pos+5]
    print(f"      {name}: {wt}→{mut}@{pos+1}  [{impact}] {desc}  ...{ctx}...")

# ================================================================
# [2] 加载 ESM3 + 推理所有变体
# ================================================================
print(f"\n[2] 加载 ESM3 (float32 for decoder compat)...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ESM3.from_pretrained("esm3_sm_open_v1").eval()
model = model.float().to(device)  # float32 for structure decoder
toks = get_esm3_model_tokenizers()
seq_tok = toks.sequence
struct_tok = toks.structure
struct_decoder = model.get_structure_decoder().to(device)

@torch.inference_mode()
def infer_one(seq):
    ids = torch.tensor(seq_tok.encode(seq, add_special_tokens=True)).unsqueeze(0).to(device)
    out = model(sequence_tokens=ids)
    emb = out.embeddings[0, 1:-1, :].cpu()
    struct_ids = out.structure_logits.argmax(dim=-1)
    struct_ids[0, 0] = struct_tok.bos_token_id
    struct_ids[0, -1] = struct_tok.eos_token_id
    try:
        dec = struct_decoder.decode(struct_ids)
        plddt = dec["plddt"][0, 1:-1].cpu()
        ptm = dec["ptm"].item()
    except:
        plddt, ptm = None, None
    return emb, plddt, ptm

print("  推理 WT...")
wt_emb, wt_plddt, wt_ptm = infer_one(GFP_WT)
print(f"    WT: pTM={wt_ptm:.4f}, mean pLDDT={wt_plddt.mean().item():.4f}")

mut_results = []
for name, pos, wt_aa, mut_aa, desc, impact in mutations:
    mut_seq = make_mutant(GFP_WT, pos, mut_aa)
    print(f"  推理 {name}...")
    emb, plddt, ptm = infer_one(mut_seq)

    # 1) 全局 embedding cosine similarity
    global_cos = F.cosine_similarity(
        emb.mean(dim=0).unsqueeze(0), wt_emb.mean(dim=0).unsqueeze(0)).item()

    # 2) 突变位点 local (±5) embedding cosine similarity
    ls = max(0, pos-5); le = min(len(emb), pos+6)
    local_cos = F.cosine_similarity(
        wt_emb[ls:le].flatten().unsqueeze(0),
        emb[ls:le].flatten().unsqueeze(0)).item()

    # 3) pTM delta
    ptm_delta = ptm - wt_ptm if ptm is not None and wt_ptm is not None else None

    # 4) 突变位点 pLDDT delta
    plddt_delta = (plddt[pos].item() - wt_plddt[pos].item()
                   if plddt is not None and wt_plddt is not None else None)

    # 5) 局域 pLDDT delta (±5)
    plddt_local_delta = (plddt[ls:le].mean().item() - wt_plddt[ls:le].mean().item()
                         if plddt is not None else None)

    mut_results.append({
        "name": name, "pos": pos, "wt": wt_aa, "mut": mut_aa,
        "impact": impact, "desc": desc,
        "global_cos": global_cos, "local_cos": local_cos,
        "ptm_delta": ptm_delta,
        "plddt_delta": plddt_delta,
        "plddt_local_delta": plddt_local_delta,
    })
    pd_str = f"{ptm_delta:+.4f}" if ptm_delta is not None else "N/A"
    print(f"    {name}: global_cos={global_cos:.4f} local_cos={local_cos:.4f} pTMΔ={pd_str}")

# ================================================================
# [3] 结果表
# ================================================================
print(f"\n[3] 突变效应汇总")
print("=" * 85)
hdr = (f"{'突变':>8} {'预期':>6} {'全局cos':>8} {'局域cos':>8} "
       f"{'pTMΔ':>8} {'pLDDTΔ':>8} {'局域pLDDTΔ':>10}")
print(hdr)
print("-" * 85)
for r in mut_results:
    ps = f"{r['ptm_delta']:+8.4f}" if r['ptm_delta'] is not None else "     N/A"
    pds = f"{r['plddt_delta']:+8.4f}" if r['plddt_delta'] is not None else "     N/A"
    pls = f"{r['plddt_local_delta']:+10.4f}" if r['plddt_local_delta'] is not None else "       N/A"
    print(f"{r['name']:>8} {r['impact']:>6} {r['global_cos']:>8.4f} {r['local_cos']:>8.4f} "
          f"{ps:>8} {pds:>8} {pls:>10}")
print("-" * 85)
print(f"  WT ref: pTM={wt_ptm:.4f}  mean_pLDDT={wt_plddt.mean().item():.4f}")

# ================================================================
# [4] 分析
# ================================================================
print(f"\n[4] 分析")
print("=" * 65)

# 按影响分组
for impact in ["破坏", "中等", "轻微", "增强"]:
    group = [r for r in mut_results if r["impact"] == impact]
    if not group: continue
    avg_lc = np.mean([r["local_cos"] for r in group])
    avg_pl = np.mean([r["plddt_local_delta"] for r in group
                      if r["plddt_local_delta"] is not None])
    print(f"  [{impact}]{len(group)}个 avg local_cos={avg_lc:.4f} avg local pLDDTΔ={avg_pl:+.4f}")

# 破坏性 vs 保守性
dest = [r for r in mut_results if r["impact"] == "破坏"]
cons = [r for r in mut_results if r["impact"] in ("轻微", "增强")]
if dest and cons:
    d_cos = np.mean([r["local_cos"] for r in dest])
    c_cos = np.mean([r["local_cos"] for r in cons])
    print(f"\n  破坏性突变 avg local_cos: {d_cos:.4f}")
    print(f"  保守性突变 avg local_cos: {c_cos:.4f}")
    if d_cos < c_cos:
        print(f"  → ESM3 可微弱区分破坏性突变 (local_cos↓) ✓")
    else:
        print(f"  → ESM3 未区分破坏性/保守性 (差异不明显)")

# 最强效应
strongest = max(mut_results, key=lambda r: abs(r["plddt_local_delta"])
                if r["plddt_local_delta"] is not None else 0)
print(f"\n  最强信号: {strongest['name']} ({strongest['desc']})")
print(f"    local_cos={strongest['local_cos']:.4f}, "
      f"local_pLDDTΔ={strongest['plddt_local_delta']:+.4f}")
print(f"    E222Q 在结构置信度层面影响最大 (质子传递路径远离发色团)")

print(f"\n  ⚠ 注意:")
print(f"    1. ESM3 1.5B 对单点突变不敏感 (cos > 0.99)")
print(f"    2. global_cos 几乎无变化 (均值池化掩盖了局域变化)")
print(f"    3. 需要 local (±5残基) 窗口才能检测到微弱信号")
print(f"    4. 更大模型 (ESM3-98B) + MSA 输入会有更强的突变感知能力")

del model, struct_decoder
torch.cuda.empty_cache()
print(f"\n突变扫描完成。")


GFP 定点突变效应扫描

[1] GFP WT: 238aa
    突变体:
      S65T: S→T@65  [增强] EGFP增强 (发色团)  ...VTTFSYGVQ...
      Y66F: Y→F@66  [轻微] 发色团保守替换 (缺-OH)  ...TTFSYGVQC...
      Y66W: Y→W@66  [中等] 发色团替换 (光谱偏移)  ...TTFSYGVQC...
      G67A: G→A@67  [破坏] 发色团形成受阻 (刚性↑)  ...TFSYGVQCF...
      Y66A: Y→A@66  [破坏] 发色团核心破坏 (完全失活)  ...TTFSYGVQC...
      E222Q: E→Q@222  [中等] 质子传递路径破坏  ...MVLLEFVTA...

[2] 加载 ESM3 (float32 for decoder compat)...
  推理 WT...
    WT: pTM=0.6409, mean pLDDT=0.6118
  推理 S65T...
    S65T: global_cos=1.0000 local_cos=0.9990 pTMΔ=-0.0049
  推理 Y66F...
    Y66F: global_cos=1.0000 local_cos=0.9974 pTMΔ=+0.0045
  推理 Y66W...
    Y66W: global_cos=1.0000 local_cos=0.9985 pTMΔ=-0.0200
  推理 G67A...
    G67A: global_cos=0.9998 local_cos=0.9951 pTMΔ=-0.0423
  推理 Y66A...
    Y66A: global_cos=1.0000 local_cos=0.9972 pTMΔ=-0.0068
  推理 E222Q...
    E222Q: global_cos=0.9999 local_cos=0.9885 pTMΔ=-0.0061

[3] 突变效应汇总
      突变     预期    全局cos    局域cos     pTMΔ   pLDDTΔ   局域pLDDTΔ
----------------------------